# MCS Case Study — Notebook 2: Split pool generator (Random vs Stratified)

This notebook generates a **pool of partitions** to quantify **performance fragility** with respect to minimal split decisions.

**Input**
- `dataset.yaml` (from Notebook 1)
- `amp_raw_10k.csv` (raw subset)

**Output**
- `splits/random/seed_<SEED>/` and `splits/stratified/seed_<SEED>/` folders containing:
  - `train_idx.npy`, `val_idx.npy`, `test_idx.npy`
  - `split.yaml` (split contract / manifest)
  - optional convenience CSVs: `train.csv`, `val.csv`, `test.csv`

We generate:
- **10** random splits (unstratified), each with a different seed
- **10** stratified splits using the **same seeds** (paired comparison)

> No modelling happens here. This notebook only produces split artifacts + split YAMLs.

In [1]:
# --- Config ---
from pathlib import Path

In [2]:
# Point these to your dataset builder outputs
DATASET_DIR = Path("../../demonstrative_applications/case_demo_01/")  # folder produced by notebook 1
RAW_CSV = DATASET_DIR / "dataset/amp_raw_10k.csv"
DATASET_YAML = DATASET_DIR / "schemas/dataset.yaml"

# Where to write split pools
SPLITS_ROOT = DATASET_DIR / "splits"

SEQ_COL = "sequence"
TARGET_COL = "Antimicrobial"

# Fractions
TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10

# Pool of seeds (paired across strategies)
SEEDS = [1, 7, 13, 21, 42, 77, 101, 1337, 2024, 9001]

# Whether to also export split CSVs (in addition to indices)
EXPORT_CSVS = True

assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9
SPLITS_ROOT.mkdir(parents=True, exist_ok=True)

print("Dataset dir:", DATASET_DIR.resolve())
print("Splits root:", SPLITS_ROOT.resolve())

Dataset dir: /home/david/Desktop/umag_projects/MinimalCommunityStandar_v0.1/demonstrative_applications/case_demo_01
Splits root: /home/david/Desktop/umag_projects/MinimalCommunityStandar_v0.1/demonstrative_applications/case_demo_01/splits


In [3]:
import pandas as pd
import yaml

# Load dataset metadata
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset_manifest = yaml.safe_load(f)

dataset_id = dataset_manifest.get("dataset", {}).get("id", "unknown_dataset_id")
print("dataset_id:", dataset_id)

# Load raw data
df = pd.read_csv(RAW_CSV)
assert SEQ_COL in df.columns, f"Missing column: {SEQ_COL}"
assert TARGET_COL in df.columns, f"Missing column: {TARGET_COL}"

# Basic sanity: target must be binary int
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").astype("Int64")
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)
assert set(df[TARGET_COL].unique()).issubset({0, 1})

print("n =", len(df))
print("label counts:\n", df[TARGET_COL].value_counts())
df.head()

dataset_id: amp_case_study_v1
n = 10000
label counts:
 Antimicrobial
0    6021
1    3979
Name: count, dtype: int64


,sequence,Antimicrobial
0,MASHPNLRRSENSFALSDTDLLPGVENFEINSVEEKLLEELGSYDE...,0
1,MMKRLIVLVLLASTLLTGCNTARGFGEDIKHLGNSISRAAS,0
2,MVTIRLARHGAKKRPFYQVVVADSRNARNGRFIERVGFFNPIASEK...,1
3,SEDRSTGNSLKDSSSFSPARY,0
4,RNIEPLNNDAMASLLSVANFKHVPAVS,0


In [4]:
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple, List
from pathlib import Path
import yaml
from datetime import datetime, timezone

@dataclass(frozen=True)
class SplitIndices:
    train: np.ndarray
    val: np.ndarray
    test: np.ndarray

def _shuffle(a: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    a = a.copy()
    rng.shuffle(a)
    return a

def make_random_split(n: int, seed: int, train_frac: float, val_frac: float, test_frac: float) -> SplitIndices:
    """Unstratified random split by indices."""
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_train = int(round(train_frac * n))
    n_val = int(round(val_frac * n))
    n_test = n - n_train - n_val

    train = idx[:n_train]
    val = idx[n_train:n_train+n_val]
    test = idx[n_train+n_val:]

    # Shuffle within each split for convenience
    train = _shuffle(train, rng)
    val = _shuffle(val, rng)
    test = _shuffle(test, rng)

    assert len(train) + len(val) + len(test) == n
    return SplitIndices(train=train, val=val, test=test)

def make_stratified_split(y: np.ndarray, seed: int, train_frac: float, val_frac: float, test_frac: float) -> SplitIndices:
    """Stratified random split preserving label proportions."""
    rng = np.random.default_rng(seed)
    indices = np.arange(len(y))

    train_parts, val_parts, test_parts = [], [], []
    for label in [0, 1]:
        idx = indices[y == label]
        rng.shuffle(idx)

        n = len(idx)
        n_train = int(round(train_frac * n))
        n_val = int(round(val_frac * n))
        n_test = n - n_train - n_val

        train_parts.append(idx[:n_train])
        val_parts.append(idx[n_train:n_train+n_val])
        test_parts.append(idx[n_train+n_val:])

    train = np.concatenate(train_parts) if train_parts else np.array([], dtype=int)
    val = np.concatenate(val_parts) if val_parts else np.array([], dtype=int)
    test = np.concatenate(test_parts) if test_parts else np.array([], dtype=int)

    # Shuffle within each split
    train = _shuffle(train, rng)
    val = _shuffle(val, rng)
    test = _shuffle(test, rng)

    assert len(train) + len(val) + len(test) == len(y)
    return SplitIndices(train=train, val=val, test=test)

def write_split_artifacts(
    df: "pd.DataFrame",
    split: SplitIndices,
    out_dir: Path,
    dataset_id: str,
    strategy: str,
    stratified: bool,
    seed: int,
    target_col: str,
    train_frac: float,
    val_frac: float,
    test_frac: float,
    export_csvs: bool = True,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    # Indices
    np.save(out_dir / "train_idx.npy", split.train)
    np.save(out_dir / "val_idx.npy", split.val)
    np.save(out_dir / "test_idx.npy", split.test)

    # Optional CSVs
    if export_csvs:
        df.iloc[split.train].to_csv(out_dir / "train.csv", index=False)
        df.iloc[split.val].to_csv(out_dir / "val.csv", index=False)
        df.iloc[split.test].to_csv(out_dir / "test.csv", index=False)

    # Split YAML (contract)
    split_manifest = {
        "mcs_version": dataset_manifest.get("mcs_version", "0.1"),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "split": {
            "id": f"{dataset_id}__{strategy}__seed_{seed}",
            "dataset_id": dataset_id,
            "strategy": strategy,
            "stratified": bool(stratified),
            "seed": int(seed),
            "target": target_col,
            "fractions": {"train": float(train_frac), "val": float(val_frac), "test": float(test_frac)},
            "artifacts": {
                "train_idx": "train_idx.npy",
                "val_idx": "val_idx.npy",
                "test_idx": "test_idx.npy",
                **({"train_csv": "train.csv", "val_csv": "val.csv", "test_csv": "test.csv"} if export_csvs else {}),
            },
            "statistics": {
                "n_total": int(len(df)),
                "n_train": int(len(split.train)),
                "n_val": int(len(split.val)),
                "n_test": int(len(split.test)),
                "label_counts": {
                    "train": df.iloc[split.train][target_col].value_counts().to_dict(),
                    "val": df.iloc[split.val][target_col].value_counts().to_dict(),
                    "test": df.iloc[split.test][target_col].value_counts().to_dict(),
                },
            },
            "notes": (
                "Split pool member for fragility analysis. "
                "Paired seeds across random and stratified variants."
            ),
        },
    }

    with open(out_dir / "split.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(split_manifest, f, sort_keys=False, allow_unicode=True)

def summarize_split(df: "pd.DataFrame", split: SplitIndices, target_col: str) -> Dict[str, Dict[int, int]]:
    return {
        "train": df.iloc[split.train][target_col].value_counts().to_dict(),
        "val": df.iloc[split.val][target_col].value_counts().to_dict(),
        "test": df.iloc[split.test][target_col].value_counts().to_dict(),
    }

In [5]:
# --- Generate split pools ---
import pandas as pd

y = df[TARGET_COL].to_numpy()

rows: List[Dict[str, object]] = []

for seed in SEEDS:
    # 1) Random (unstratified)
    split_r = make_random_split(
        n=len(df),
        seed=seed,
        train_frac=TRAIN_FRAC,
        val_frac=VAL_FRAC,
        test_frac=TEST_FRAC,
    )
    out_r = SPLITS_ROOT / "random" / f"seed_{seed}"
    write_split_artifacts(
        df=df, split=split_r, out_dir=out_r,
        dataset_id=dataset_id,
        strategy="random",
        stratified=False,
        seed=seed,
        target_col=TARGET_COL,
        train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
        export_csvs=EXPORT_CSVS,
    )
    rows.append({"strategy": "random", "seed": seed, **{f"{k}_counts": v for k, v in summarize_split(df, split_r, TARGET_COL).items()}})

    # 2) Random + stratified
    split_s = make_stratified_split(
        y=y,
        seed=seed,
        train_frac=TRAIN_FRAC,
        val_frac=VAL_FRAC,
        test_frac=TEST_FRAC,
    )
    out_s = SPLITS_ROOT / "stratified" / f"seed_{seed}"
    write_split_artifacts(
        df=df, split=split_s, out_dir=out_s,
        dataset_id=dataset_id,
        strategy="random",
        stratified=True,
        seed=seed,
        target_col=TARGET_COL,
        train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
        export_csvs=EXPORT_CSVS,
    )
    rows.append({"strategy": "stratified", "seed": seed, **{f"{k}_counts": v for k, v in summarize_split(df, split_s, TARGET_COL).items()}})

summary = pd.DataFrame(rows).sort_values(["seed", "strategy"]).reset_index(drop=True)
print("Wrote split pools under:", SPLITS_ROOT)
summary

Wrote split pools under: ../../demonstrative_applications/case_demo_01/splits


,strategy,seed,train_counts,val_counts,test_counts
0,random,1,"{0: 4827, 1: 3173}","{0: 580, 1: 420}","{0: 614, 1: 386}"
1,stratified,1,"{0: 4817, 1: 3183}","{0: 602, 1: 398}","{0: 602, 1: 398}"
2,random,7,"{0: 4808, 1: 3192}","{0: 603, 1: 397}","{0: 610, 1: 390}"
3,stratified,7,"{0: 4817, 1: 3183}","{0: 602, 1: 398}","{0: 602, 1: 398}"
4,random,13,"{0: 4821, 1: 3179}","{0: 604, 1: 396}","{0: 596, 1: 404}"
5,stratified,13,"{0: 4817, 1: 3183}","{0: 602, 1: 398}","{0: 602, 1: 398}"
6,random,21,"{0: 4819, 1: 3181}","{0: 584, 1: 416}","{0: 618, 1: 382}"
7,stratified,21,"{0: 4817, 1: 3183}","{0: 602, 1: 398}","{0: 602, 1: 398}"
8,random,42,"{0: 4827, 1: 3173}","{0: 585, 1: 415}","{0: 609, 1: 391}"
9,stratified,42,"{0: 4817, 1: 3183}","{0: 602, 1: 398}","{0: 602, 1: 398}"


In [6]:
# --- Quick integrity checks ---
from pathlib import Path
import numpy as np

def check_one(folder: Path):
    tr = np.load(folder / "train_idx.npy")
    va = np.load(folder / "val_idx.npy")
    te = np.load(folder / "test_idx.npy")

    all_idx = np.concatenate([tr, va, te])
    assert len(all_idx) == len(df), f"Wrong total size in {folder}"
    assert len(np.unique(all_idx)) == len(df), f"Duplicate indices in {folder}"
    assert set(all_idx.tolist()) == set(range(len(df))), f"Missing indices in {folder}"

for seed in SEEDS:
    check_one(SPLITS_ROOT / "random" / f"seed_{seed}")
    check_one(SPLITS_ROOT / "stratified" / f"seed_{seed}")

print("All split artifacts passed integrity checks")

All split artifacts passed integrity checks
